In [1]:
import os
import rasterio
from importlib_resources import files
from pathlib import Path
from tqdm import tqdm

from beak.experimental.raster_processing import unify_raster_grids
from beak.experimental.io import save_raster, create_file_list, check_path


**User inputs**
1. Select the root folder where the rasters are stored.
2. Then, go down to select the subfolders that contain the rasters to be unified.<p>

The script will search for all rasters and store them in relative paths.

In [2]:
BASE_PATH = files("beak.data")

TARGET_RES = 2240
TARGET_EPSG = 102008
SPATIAL_EXTENT = "us_cont"

PATH_ROOT = BASE_PATH / "RAW" / "datacube_lawley" / "exports"
BASE_RASTER = BASE_PATH / "BASE_RASTERS" / f"EPSG_{TARGET_EPSG}_RES_{TARGET_RES}_{SPATIAL_EXTENT}.tif"
PATH_INPUT = PATH_ROOT / "national_us_canada_4326_0_02"

# Export options
PATH_EXPORT = BASE_PATH / "PROCESSED" / f"national_{SPATIAL_EXTENT}_{TARGET_EPSG}_{TARGET_RES}" / "unified_lawley"

# Optional example export
CURRENT_DIR = Path(os.getcwd()) / PATH_EXPORT.name

# Set selected output path
OUT_FOLDER = PATH_EXPORT

print(f"Input folder: {PATH_INPUT}")
print(f"Export folder: {OUT_FOLDER}")

Input folder: S:\Projekte\20230082_DARPA_CriticalMAAS_TA3\Bearbeitung\GitHub\beak-ta3\src\beak\data\RAW\datacube_lawley\exports\national_us_canada_4326_0_02
Export folder: S:\Projekte\20230082_DARPA_CriticalMAAS_TA3\Bearbeitung\GitHub\beak-ta3\src\beak\data\PROCESSED\national_us_cont_102008_2240\unified_lawley


Select subfolders to be scaled.

In [3]:
folders = os.listdir(PATH_INPUT)
for folder in folders:
  if os.path.isdir(os.path.join(PATH_INPUT, folder)):
    print(folder)


categorical
ground_truth
numerical
numerical_imputed
numerical_imputed_scaled_standard
numerical_scaled_standard


**Run unification**

In [5]:
from beak.experimental.io import create_file_folder_list
folders, files = create_file_folder_list(PATH_INPUT)

exclude_list = ["numerical_imputed",
                "numerical_imputed_scaled_standard",
                "numerical_scaled_standard"]

folders = [folder for folder in folders if folder.name not in exclude_list]
files = [file for file in files if file.parent.name not in exclude_list]

print(f"Total of folders found: {len(folders)}")
print(f"Total of files found: {len(files)}")

Total of folders found: 49
Total of files found: 820


In [6]:
import multiprocessing as mp

for folder in tqdm(folders):
  folder_relative = os.path.relpath(folder, PATH_INPUT)
  files = create_file_list(folder)
  
  if len(files) > 0:
    result = unify_raster_grids(BASE_RASTER, files, resampling_mode="auto", same_extent=True, same_shape=True, n_workers=mp.cpu_count())
    
    output_folder = OUT_FOLDER / str(folder_relative)
    for i, file in enumerate(files):
      raster = rasterio.open(file)
      out_path = output_folder / file.name
      check_path(Path(os.path.dirname(out_path)))
      save_raster(out_path, array=result[i][0], metadata=result[i][1], dtype=raster.meta["dtype"], overwrite=True)
  

100%|██████████| 49/49 [02:00<00:00,  2.45s/it]
